In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

import random
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import (RobertaTokenizer, GPT2LMHeadModel,
                          GPT2Config)
import timm
import Levenshtein
from tqdm import tqdm
from collections import defaultdict

# =====================================================================
# 1. CONFIGURATION
# =====================================================================
BASE_DATA_DIR   = "/home/mca24-26/ocr/dataset/iam/iam_words"
WORDS_TXT_FILE  = os.path.join(BASE_DATA_DIR, "words.txt")
CHECKPOINT_PATH = "/home/mca24-26/ocr/iam_htr_distilgpt2.pth"

IMG_HEIGHT  = 96
IMG_WIDTH   = 384
MAX_SEQ_LEN = 25
BEAM_SIZE   = 5
D_MODEL     = 768       # DistilGPT-2 also uses 768 — no change needed
BATCH_SIZE  = 16

# =====================================================================
# 2. DATASET  (unchanged)
# =====================================================================
class IAMWordsDataset(Dataset):
    def __init__(self, data_dir, words_txt_path, processor,
                 max_target_length=MAX_SEQ_LEN):
        self.data_dir          = data_dir
        self.processor         = processor
        self.max_target_length = max_target_length
        self.samples           = []
        self._parse_words_txt(words_txt_path)

    def _parse_words_txt(self, path):
        if not os.path.exists(path):
            print(f"Error: {path} not found.")
            return
        with open(path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) < 9:
                    continue
                word_id       = parts[0]
                if parts[1] != "ok":
                    continue
                transcription = " ".join(parts[8:])
                if not transcription.strip():
                    continue
                id_parts = word_id.split('-')
                img_path = os.path.join(
                    self.data_dir, "words",
                    id_parts[0],
                    f"{id_parts[0]}-{id_parts[1]}",
                    f"{word_id}.png"
                )
                if os.path.exists(img_path):
                    self.samples.append((img_path, transcription))
        print(f"Registered {len(self.samples)} samples.")

    def __len__(self):
        return len(self.samples)


def get_iam_dataloaders(data_dir, words_txt_path,
                        processor, batch_size=BATCH_SIZE):
    train_transform = T.Compose([
        T.Resize((IMG_HEIGHT, IMG_WIDTH)),
        T.RandomApply([T.ColorJitter(
            brightness=0.3, contrast=0.3)], p=0.5),
        T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.3),
        T.RandomRotation(degrees=3),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
    ])
    val_transform = T.Compose([
        T.Resize((IMG_HEIGHT, IMG_WIDTH)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
    ])

    full_dataset = IAMWordsDataset(
        data_dir, words_txt_path, processor=processor
    )
    total      = len(full_dataset)
    train_size = int(0.9 * total)
    generator  = torch.Generator().manual_seed(42)
    indices    = torch.randperm(total, generator=generator).tolist()

    train_samples = [full_dataset.samples[i]
                     for i in indices[:train_size]]
    val_samples   = [full_dataset.samples[i]
                     for i in indices[train_size:]]

    class SplitDataset(Dataset):
        def __init__(self, samples, processor,
                     transform, max_target_length=MAX_SEQ_LEN):
            self.samples           = samples
            self.processor         = processor
            self.transform         = transform
            self.max_target_length = max_target_length

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx):
            img_path, text = self.samples[idx]
            try:
                image = Image.open(img_path).convert('RGB')
            except Exception:
                image = Image.new('RGB',
                                  (IMG_WIDTH, IMG_HEIGHT), 255)
            if self.transform:
                image = self.transform(image)
            labels = self.processor(
                text,
                padding='max_length',
                max_length=self.max_target_length,
                truncation=True,
                return_tensors="pt"
            ).input_ids.squeeze(0)
            labels = torch.where(
                labels == self.processor.pad_token_id,
                torch.tensor(-100), labels
            )
            return image, labels, text

    train_loader = DataLoader(
        SplitDataset(train_samples, processor, train_transform),
        batch_size=batch_size, shuffle=True,
        num_workers=4, drop_last=True
    )
    val_loader = DataLoader(
        SplitDataset(val_samples, processor, val_transform),
        batch_size=batch_size, shuffle=False, num_workers=4
    )
    return train_loader, val_loader


# =====================================================================
# 3. ARCHITECTURE
# =====================================================================
class LocalizationNetwork(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.num_control_points = num_control_points
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(True),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(True),
            nn.Linear(256, num_control_points * 2)
        )
        self.fc[-1].weight.data.zero_()
        self.fc[-1].bias.data.zero_()

    def forward(self, x):
        B   = x.size(0)
        pts = self.fc(self.conv(x).view(B, -1))
        return pts.view(B, self.num_control_points, 2)


class TPSSpatialTransformer(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.localization = LocalizationNetwork(num_control_points)

    def forward(self, x):
        B              = x.size(0)
        cp             = self.localization(x)
        cx             = cp[:, :, 0].mean(dim=1)
        cy             = cp[:, :, 1].mean(dim=1)
        theta          = torch.zeros(B, 2, 3, device=x.device)
        theta[:, 0, 0] = 1.0
        theta[:, 1, 1] = 1.0
        theta[:, 0, 2] = torch.tanh(cx) * 0.05
        theta[:, 1, 2] = torch.tanh(cy) * 0.05
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        return F.grid_sample(x, grid, align_corners=False,
                             padding_mode='border')


class SwinEncoder(nn.Module):
    def __init__(self, d_model=D_MODEL):
        super().__init__()
        self.swin = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            img_size=(IMG_HEIGHT, IMG_WIDTH),
            strict_img_size=False,
        )
        self.proj = nn.Linear(512, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        features   = self.swin(x)
        feat       = features[-2]
        B, H, W, C = feat.shape
        feat       = feat.view(B, H*W, C)
        return self.norm(self.proj(feat))


# =====================================================================
# CHANGE 1 — GPT2Decoder now loads "distilgpt2" instead of "gpt2"
#
# What changed:
#   - from_pretrained("gpt2")       →  from_pretrained("distilgpt2")
#   - n_layer: 12                   →  6   (half the layers)
#   - Parameters: ~117M             →  ~82M
#
# What did NOT change:
#   - n_embd stays 768              (BiLSTM output still matches)
#   - n_head stays 12
#   - add_cross_attention = True    (same cross-attn setup)
#   - PRETRAINED_VOCAB = 50257      (same vocab copying logic)
#   - forward() method              (identical)
#   - All LLRD group names          (identical)
# =====================================================================
class GPT2Decoder(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL):
        super().__init__()
        print("Loading DistilGPT-2 pretrained weights...")  # updated print

        PRETRAINED_VOCAB = 50257

        # CHANGE: "gpt2" → "distilgpt2"
        config                     = GPT2Config.from_pretrained("distilgpt2")
        config.add_cross_attention = True
        config.vocab_size          = vocab_size

        self.decoder = GPT2LMHeadModel(config)

        # CHANGE: load from "distilgpt2" checkpoint
        pretrained      = GPT2LMHeadModel.from_pretrained("distilgpt2")
        pretrained_dict = pretrained.state_dict()

        mismatch_keys = {
            "transformer.wte.weight",
            "lm_head.weight"
        }
        filtered_dict = {
            k: v for k, v in pretrained_dict.items()
            if k not in mismatch_keys
        }

        load_result = self.decoder.load_state_dict(
            filtered_dict, strict=False
        )
        print(f"  Loaded: {len(filtered_dict)} weight tensors")
        print(f"  Missing (new layers): {len(load_result.missing_keys)}")

        with torch.no_grad():
            self.decoder.transformer.wte.weight \
                .data[:PRETRAINED_VOCAB].copy_(
                    pretrained_dict["transformer.wte.weight"]
                )
            self.decoder.lm_head.weight \
                .data[:PRETRAINED_VOCAB].copy_(
                    pretrained_dict["lm_head.weight"]
                )

        del pretrained, pretrained_dict
        print(f"DistilGPT-2 weights loaded successfully.")   # updated print
        print(f"  Layers : 6 (vs 12 in GPT-2 Small)")       # new info line
        print(f"  Vocab: {PRETRAINED_VOCAB} pretrained "
              f"+ {vocab_size - PRETRAINED_VOCAB} new tokens")
        print(f"  Cross-attention layers: randomly initialized")

    def forward(self, input_ids,
                encoder_hidden_states=None, labels=None):
        return self.decoder(
            input_ids             = input_ids,
            encoder_hidden_states = encoder_hidden_states,
            labels                = labels
        )


# =====================================================================
# CompleteHTRPipeline  (unchanged — d_model=768 still matches)
# =====================================================================
class CompleteHTRPipeline(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL,
                 num_control_points=16):
        super().__init__()
        self.tps_stn      = TPSSpatialTransformer(num_control_points)
        self.swin_encoder = SwinEncoder(d_model=d_model)
        self.bilstm       = nn.LSTM(
            input_size    = d_model,
            hidden_size   = d_model // 2,
            num_layers    = 2,
            batch_first   = True,
            bidirectional = True,
            dropout       = 0.3
        )
        self.gpt2_decoder = GPT2Decoder(
            vocab_size=vocab_size, d_model=d_model
        )

    def _extract_visual_memory(self, images):
        rectified = self.tps_stn(images)
        swin_feat = self.swin_encoder(rectified)
        memory, _ = self.bilstm(swin_feat)
        return memory.contiguous()

    def forward(self, images, target_ids, criterion=None):
        memory = self._extract_visual_memory(images)

        dec_input = target_ids[:, :-1].clone()
        dec_input = torch.where(
            dec_input == -100,
            torch.ones_like(dec_input),
            dec_input
        )
        labels = target_ids[:, 1:].clone()

        outputs = self.gpt2_decoder(
            input_ids             = dec_input,
            encoder_hidden_states = memory,
            labels                = None
        )
        logits = outputs.logits

        if criterion is not None:
            return criterion(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1)
            )
        return F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1),
            ignore_index=-100
        )

    @torch.no_grad()
    def generate(self, images, max_length=MAX_SEQ_LEN,
                 bos_token_id=0, eos_token_id=2,
                 beam_size=BEAM_SIZE):
        device = images.device
        B      = images.size(0)
        memory = self._extract_visual_memory(images)

        if beam_size == 1:
            generated = torch.full(
                (B, 1), bos_token_id,
                dtype=torch.long, device=device
            )
            finished = torch.zeros(B, dtype=torch.bool, device=device)
            for _ in range(max_length - 1):
                out         = self.gpt2_decoder(
                    input_ids             = generated,
                    encoder_hidden_states = memory,
                    labels                = None
                )
                next_tokens = out.logits[:, -1, :].argmax(
                    dim=-1, keepdim=True
                )
                next_tokens = next_tokens.masked_fill(
                    finished.unsqueeze(1), eos_token_id
                )
                generated   = torch.cat([generated, next_tokens], dim=1)
                finished   |= (next_tokens.squeeze(1) == eos_token_id)
                if finished.all():
                    break
            return generated[:, 1:]

        all_results = []
        for b in range(B):
            mem       = memory[b:b+1]
            beams     = [(0.0, [bos_token_id])]
            completed = []

            for _ in range(max_length - 1):
                candidates = []
                for score, tokens in beams:
                    if tokens[-1] == eos_token_id:
                        completed.append((score, tokens))
                        continue
                    inp = torch.tensor(
                        [tokens], dtype=torch.long, device=device
                    )
                    out      = self.gpt2_decoder(
                        input_ids             = inp,
                        encoder_hidden_states = mem,
                        labels                = None
                    )
                    log_prob = F.log_softmax(
                        out.logits[0, -1], dim=-1
                    )
                    top_lp, top_id = log_prob.topk(beam_size)
                    for lp, tid in zip(top_lp.tolist(),
                                       top_id.tolist()):
                        candidates.append(
                            (score + lp, tokens + [tid])
                        )

                if not candidates:
                    break

                candidates.sort(key=lambda x: x[0], reverse=True)
                beams = []
                for score, tokens in candidates[:beam_size * 2]:
                    if tokens[-1] == eos_token_id:
                        completed.append((score, tokens))
                    else:
                        beams.append((score, tokens))
                    if len(beams) == beam_size:
                        break

                if not beams:
                    break

            all_beams        = completed if completed else beams
            _, best_tokens   = max(all_beams, key=lambda x: x[0])
            result           = torch.tensor(
                best_tokens[1:], dtype=torch.long, device=device
            )
            all_results.append(result)

        max_len = max(r.size(0) for r in all_results)
        padded  = torch.full(
            (B, max_len), eos_token_id,
            dtype=torch.long, device=device
        )
        for i, r in enumerate(all_results):
            padded[i, :r.size(0)] = r
        return padded


# =====================================================================
# 4. AGENTIC VERIFICATION  (unchanged)
# =====================================================================
class AgenticVerificationModule:
    def __init__(self, words_txt_file=None):
        self.lexicon  = {}
        self.freq_max = 1
        if words_txt_file and os.path.exists(words_txt_file):
            self._build_lexicon(words_txt_file)

    def _build_lexicon(self, file_path):
        freq = defaultdict(int)
        with open(file_path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) >= 9 and parts[1] == "ok":
                    word = " ".join(parts[8:]).strip().lower()
                    if word:
                        freq[word] += 1
        self.lexicon  = dict(freq)
        self.freq_max = max(freq.values()) if freq else 1
        print(f"Lexicon: {len(self.lexicon)} words | "
              f"Max freq: {self.freq_max}")

    def verify_and_correct(self, text_output,
                           top_logprob=None,
                           logprob_threshold=-0.3):
        cleaned = text_output.strip().lower()

        if (not cleaned
                or cleaned in self.lexicon
                or len(cleaned) <= 2
                or any(c.isdigit() for c in cleaned)):
            return text_output

        if text_output[0].isupper() and len(text_output) > 1:
            if text_output[1:].islower():
                if cleaned in self.lexicon:
                    return text_output

        if top_logprob is not None:
            avg_lp = sum(top_logprob) / max(len(top_logprob), 1)
            if avg_lp > logprob_threshold:
                return text_output

        best_candidate = None
        best_score     = -float('inf')
        target_len     = len(cleaned)

        for word, freq in self.lexicon.items():
            if abs(len(word) - target_len) > 1:
                continue
            dist = Levenshtein.distance(cleaned, word)
            if dist > 1:
                continue
            freq_score = freq / self.freq_max
            score      = freq_score - (dist * 0.3)
            if score > best_score:
                best_score     = score
                best_candidate = word

        if best_candidate is None:
            return text_output

        if text_output.isupper():
            return best_candidate.upper()
        elif len(text_output) > 0 and text_output[0].isupper():
            return best_candidate.capitalize()
        return best_candidate


# =====================================================================
# 5. METRICS & EARLY STOPPING  (unchanged)
# =====================================================================
def calculate_metrics(predictions, targets):
    if not targets:
        return {"CER": 0.0, "WER": 0.0}
    total_cer   = 0.0
    wrong_words = 0
    for pred, target in zip(predictions, targets):
        p = pred.strip()
        t = target.strip()
        d = Levenshtein.distance(p, t)
        if len(t) > 0:
            total_cer += d / len(t)
        elif len(p) > 0:
            total_cer += 1.0
        if d != 0:
            wrong_words += 1
    n = len(targets)
    return {
        "CER": round((total_cer / n) * 100, 4),
        "WER": round((wrong_words / n) * 100, 4)
    }


class EarlyStopping:
    def __init__(self, patience=8, min_delta=0.05):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best       = float('inf')
        self.early_stop = False

    def __call__(self, metric):
        if metric < self.best - self.min_delta:
            self.best    = metric
            self.counter = 0
        else:
            self.counter += 1
            print(f"EarlyStopping: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop


# =====================================================================
# 6. LLRD OPTIMIZER  (unchanged — group names are identical)
# =====================================================================
def build_llrd_optimizer(model, base_lr=5e-5,
                         decay_factor=0.75,
                         weight_decay=0.05):
    assigned = set()

    def collect(named_params, filter_fn):
        params = []
        for name, param in named_params:
            if id(param) not in assigned and filter_fn(name):
                params.append(param)
                assigned.add(id(param))
        return params

    def collect_all(named_params):
        params = []
        for name, param in named_params:
            if id(param) not in assigned:
                params.append(param)
                assigned.add(id(param))
        return params

    gpt2_crossattn = collect(
        model.gpt2_decoder.named_parameters(),
        lambda n: "crossattention" in n or "cross_attn" in n
    )
    gpt2_selfattn = collect_all(
        model.gpt2_decoder.named_parameters()
    )
    bilstm_params = collect_all(model.bilstm.named_parameters())
    swin_proj = collect(
        model.swin_encoder.named_parameters(),
        lambda n: n.startswith("proj.") or n.startswith("norm.")
    )
    swin_s4 = collect(
        model.swin_encoder.swin.named_parameters(),
        lambda n: "layers_3" in n
    )
    swin_s3 = collect(
        model.swin_encoder.swin.named_parameters(),
        lambda n: "layers_2" in n
    )
    swin_s2 = collect(
        model.swin_encoder.swin.named_parameters(),
        lambda n: "layers_1" in n
    )
    swin_s1 = collect_all(
        model.swin_encoder.swin.named_parameters()
    )
    tps_params = collect_all(model.tps_stn.named_parameters())

    lr = [
        base_lr,
        base_lr * decay_factor,
        base_lr * decay_factor**2,
        base_lr * decay_factor**3,
        base_lr * decay_factor**4,
        base_lr * decay_factor**5,
        base_lr * decay_factor**6,
        base_lr * decay_factor**7,
        base_lr * decay_factor**3,
    ]

    groups_raw = [
        (gpt2_crossattn, lr[0], "gpt2_crossattn"),
        (gpt2_selfattn,  lr[1], "gpt2_selfattn"),
        (bilstm_params,  lr[2], "bilstm"),
        (swin_proj,      lr[3], "swin_proj"),
        (swin_s4,        lr[4], "swin_stage4"),
        (swin_s3,        lr[5], "swin_stage3"),
        (swin_s2,        lr[6], "swin_stage2"),
        (swin_s1,        lr[7], "swin_stage1"),
        (tps_params,     lr[8], "tps_stn"),
    ]

    param_groups = [
        {"params": params, "lr": lrate, "name": name}
        for params, lrate, name in groups_raw
        if len(params) > 0
    ]

    total_assigned = sum(len(g["params"]) for g in param_groups)
    total_model    = sum(1 for _ in model.parameters())
    if total_assigned != total_model:
        print(f"WARNING: {total_model - total_assigned} params unassigned")

    print("\nLLRD Groups:")
    print(f"{'Name':<20} {'LR':>10} {'Params':>10}")
    print("-" * 44)
    for g in param_groups:
        n = sum(p.numel() for p in g["params"])
        print(f"{g['name']:<20} {g['lr']:>10.2e} {n/1e6:>9.2f}M")

    return torch.optim.AdamW(param_groups, weight_decay=weight_decay)


# =====================================================================
# 7. TRAINING  (unchanged)
# =====================================================================
def run_training_pipeline(epochs=60, batch_size=BATCH_SIZE,
                          base_lr=5e-5,
                          early_stopping_patience=8,
                          resume_from=None):

    print("Loading tokenizer...")
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
    print(f"BOS={tokenizer.bos_token_id} | "
          f"EOS={tokenizer.eos_token_id} | "
          f"PAD={tokenizer.pad_token_id}")

    train_loader, val_loader = get_iam_dataloaders(
        BASE_DATA_DIR, WORDS_TXT_FILE,
        tokenizer, batch_size=batch_size
    )

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    model  = CompleteHTRPipeline(
        vocab_size=tokenizer.vocab_size
    ).to(device)

    total_p = sum(p.numel() for p in model.parameters())
    print(f"Total params : {total_p/1e6:.2f}M")
    print(f"Decoder      : DistilGPT-2 (6 layers, 82M)")
    print(f"Resolution   : {IMG_HEIGHT}x{IMG_WIDTH}")
    print(f"Beam size    : {BEAM_SIZE}")

    criterion = nn.CrossEntropyLoss(
        ignore_index=-100, label_smoothing=0.1
    )

    for param in model.swin_encoder.swin.parameters():
        param.requires_grad = False
    print("Swin frozen for first 3 epochs.")

    optimizer = build_llrd_optimizer(
        model, base_lr=base_lr,
        decay_factor=0.75, weight_decay=0.05
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-7
    )

    agent_verifier = AgenticVerificationModule(
        words_txt_file=WORDS_TXT_FILE
    )
    early_stopper  = EarlyStopping(
        patience=early_stopping_patience, min_delta=0.05
    )
    best_val_wer   = float('inf')
    start_epoch    = 1
    swin_unfrozen  = False

    if resume_from and os.path.exists(resume_from):
        print(f"\nResuming from: {resume_from}")
        ckpt = torch.load(resume_from, map_location=device,
                          weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        best_val_wer = ckpt.get('best_wer', float('inf'))
        start_epoch  = ckpt.get('epoch', 1) + 1
        if start_epoch > 4:
            for param in model.swin_encoder.swin.parameters():
                param.requires_grad = True
            swin_unfrozen = True
        print(f"Resumed epoch {start_epoch} | "
              f"Best WER: {best_val_wer:.2f}%")

    print(f"\nTraining on: {device}")

    for epoch in range(start_epoch, epochs + 1):

        if epoch == 4 and not swin_unfrozen:
            for param in model.swin_encoder.swin.parameters():
                param.requires_grad = True
            optimizer = build_llrd_optimizer(
                model, base_lr=base_lr,
                decay_factor=0.75, weight_decay=0.05
            )
            scheduler = torch.optim.lr_scheduler \
                .CosineAnnealingWarmRestarts(
                    optimizer, T_0=10, T_mult=2, eta_min=1e-7
                )
            swin_unfrozen = True
            print("Swin encoder unfrozen — LLRD optimizer rebuilt.")

        # ── Train ──────────────────────────────────────────
        model.train()
        total_loss = 0.0
        pbar = tqdm(train_loader,
                    desc=f"Epoch {epoch}/{epochs} [Train]")
        for images, labels, _ in pbar:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            loss = model(images, target_ids=labels,
                         criterion=criterion)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), max_norm=1.0
            )
            optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        scheduler.step()
        avg_loss = total_loss / len(train_loader)

        lrs = {g["name"]: g["lr"]
               for g in optimizer.param_groups}
        print(f"Epoch {epoch} | Loss: {avg_loss:.4f} | "
              f"LR gpt2_cross: {lrs.get('gpt2_crossattn', 0):.2e} "
              f"| LR swin4: {lrs.get('swin_stage4', 0):.2e}")

        # ── Validate ───────────────────────────────────────
        model.eval()
        all_preds, all_targets = [], []
        first_batch_done       = False

        with torch.no_grad():
            for images, _, text_labels in tqdm(
                    val_loader,
                    desc=f"Epoch {epoch}/{epochs} [Val]"):
                images = images.to(device)
                tokens = model.generate(
                    images,
                    max_length   = MAX_SEQ_LEN,
                    bos_token_id = tokenizer.bos_token_id,
                    eos_token_id = tokenizer.eos_token_id,
                    beam_size    = 1
                )
                if not first_batch_done:
                    print(f"\n--- Debug Epoch {epoch} ---")
                    for i in range(min(3, images.size(0))):
                        raw = tokenizer.decode(
                            tokens[i], skip_special_tokens=True
                        )
                        print(f"  Target: '{text_labels[i]}' "
                              f"| Pred: '{raw.strip()}'")
                    print("---")
                    first_batch_done = True

                for i in range(images.size(0)):
                    raw      = tokenizer.decode(
                        tokens[i], skip_special_tokens=True
                    )
                    verified = agent_verifier.verify_and_correct(raw)
                    all_preds.append(verified)
                    all_targets.append(text_labels[i])

        metrics     = calculate_metrics(all_preds, all_targets)
        current_wer = metrics['WER']
        print(f"Epoch {epoch} | "
              f"CER: {metrics['CER']:.2f}% | "
              f"WER: {current_wer:.2f}%")

        if current_wer < best_val_wer:
            best_val_wer = current_wer
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_wer':             best_val_wer,
                'cer':                  metrics['CER'],
                'img_height':           IMG_HEIGHT,
                'img_width':            IMG_WIDTH,
                'decoder':              'distilgpt2',   # NEW: track which decoder
            }, CHECKPOINT_PATH)
            print(f"Checkpoint saved | WER: {best_val_wer:.2f}%")

        if early_stopper(current_wer):
            print(f"Early stopping at epoch {epoch}.")
            break

        print("=" * 70)


if __name__ == "__main__":
    run_training_pipeline(
        epochs                  = 60,
        batch_size              = BATCH_SIZE,
        base_lr                 = 5e-5,
        early_stopping_patience = 8,
        resume_from             = CHECKPOINT_PATH
    )

Loading tokenizer...


/usr/local/lib/python3.8/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


BOS=0 | EOS=2 | PAD=1
Registered 38305 samples.


/usr/local/lib/python3.8/dist-packages/timm/layers/interpolate.py:47: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at ../aten/src/ATen/native/BucketizationUtils.h:32.)
  idx_right = torch.bucketize(x, p)


Loading DistilGPT-2 pretrained weights...


...

Epoch 41 | Loss: 1.5207 | LR gpt2_cross: 4.52e-05 | LR swin4: 1.43e-05


Epoch 41/60 [Val]:   1%|▎                                  | 2/240 [00:00<00:45,  5.27it/s]


--- Debug Epoch 41 ---
  Target: 'any' | Pred: 'any'
  Target: 'any' | Pred: 'any'
  Target: 'unopposed' | Pred: 'wrapped'
---


Epoch 41/60 [Val]: 100%|█████████████████████████████████| 240/240 [00:34<00:00,  6.87it/s]

Epoch 41 | CER: 16.63% | WER: 25.45%
EarlyStopping: 8/8
Early stopping at epoch 41.


In [1]:
# TEST SCRIPT FOR IAM HTR WITH DISTILGPT‑2 (BEAM SEARCH)
import os
import random
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import RobertaTokenizer, GPT2LMHeadModel, GPT2Config
import timm
import Levenshtein
from tqdm import tqdm
from collections import defaultdict

# =====================================================================
# 1. CONFIGURATION (must match training)
# =====================================================================
BASE_DATA_DIR   = "/home/mca24-26/ocr/dataset/iam/iam_words"
WORDS_TXT_FILE  = os.path.join(BASE_DATA_DIR, "words.txt")
CHECKPOINT_PATH = "/home/mca24-26/ocr/iam_htr_distilgpt2.pth"

IMG_HEIGHT  = 96
IMG_WIDTH   = 384
MAX_SEQ_LEN = 25
BEAM_SIZE   = 5            # beam size for testing
BATCH_SIZE  = 16           # can be larger for inference
D_MODEL     = 768

# =====================================================================
# 2. DATASET LOADER (reuse training code)
# =====================================================================
class IAMWordsDataset(Dataset):
    def __init__(self, data_dir, words_txt_path, processor,
                 max_target_length=MAX_SEQ_LEN):
        self.data_dir          = data_dir
        self.processor         = processor
        self.max_target_length = max_target_length
        self.samples           = []
        self._parse_words_txt(words_txt_path)

    def _parse_words_txt(self, path):
        if not os.path.exists(path):
            print(f"Error: {path} not found.")
            return
        with open(path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) < 9:
                    continue
                word_id       = parts[0]
                if parts[1] != "ok":
                    continue
                transcription = " ".join(parts[8:])
                if not transcription.strip():
                    continue
                id_parts = word_id.split('-')
                img_path = os.path.join(
                    self.data_dir, "words",
                    id_parts[0],
                    f"{id_parts[0]}-{id_parts[1]}",
                    f"{word_id}.png"
                )
                if os.path.exists(img_path):
                    self.samples.append((img_path, transcription))
        print(f"Registered {len(self.samples)} samples.")

    def __len__(self):
        return len(self.samples)

def get_iam_test_loader(data_dir, words_txt_path, processor, batch_size=32):
    """Creates a DataLoader for the validation split (10%) used in training."""
    val_transform = T.Compose([
        T.Resize((IMG_HEIGHT, IMG_WIDTH)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
    ])

    full_dataset = IAMWordsDataset(data_dir, words_txt_path, processor=processor)
    total = len(full_dataset)
    train_size = int(0.9 * total)
    generator = torch.Generator().manual_seed(42)
    indices = torch.randperm(total, generator=generator).tolist()
    val_indices = indices[train_size:]   # validation split (same as training)

    class SplitDataset(Dataset):
        def __init__(self, samples, processor, transform, max_target_length=MAX_SEQ_LEN):
            self.samples = samples
            self.processor = processor
            self.transform = transform
            self.max_target_length = max_target_length

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx):
            img_path, text = self.samples[idx]
            try:
                image = Image.open(img_path).convert('RGB')
            except Exception:
                image = Image.new('RGB', (IMG_WIDTH, IMG_HEIGHT), 255)
            if self.transform:
                image = self.transform(image)
            labels = self.processor(
                text,
                padding='max_length',
                max_length=self.max_target_length,
                truncation=True,
                return_tensors="pt"
            ).input_ids.squeeze(0)
            labels = torch.where(
                labels == self.processor.pad_token_id,
                torch.tensor(-100), labels
            )
            return image, labels, text

    val_samples = [full_dataset.samples[i] for i in val_indices]
    val_dataset = SplitDataset(val_samples, processor, val_transform)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    return val_loader

# =====================================================================
# 3. MODEL DEFINITION (same as training)
# =====================================================================
class LocalizationNetwork(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.num_control_points = num_control_points
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(True),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(True),
            nn.Linear(256, num_control_points * 2)
        )
        self.fc[-1].weight.data.zero_()
        self.fc[-1].bias.data.zero_()

    def forward(self, x):
        B = x.size(0)
        pts = self.fc(self.conv(x).view(B, -1))
        return pts.view(B, self.num_control_points, 2)

class TPSSpatialTransformer(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.localization = LocalizationNetwork(num_control_points)

    def forward(self, x):
        B = x.size(0)
        cp = self.localization(x)
        cx = cp[:, :, 0].mean(dim=1)
        cy = cp[:, :, 1].mean(dim=1)
        theta = torch.zeros(B, 2, 3, device=x.device)
        theta[:, 0, 0] = 1.0
        theta[:, 1, 1] = 1.0
        theta[:, 0, 2] = torch.tanh(cx) * 0.05
        theta[:, 1, 2] = torch.tanh(cy) * 0.05
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        return F.grid_sample(x, grid, align_corners=False, padding_mode='border')

class SwinEncoder(nn.Module):
    def __init__(self, d_model=D_MODEL):
        super().__init__()
        self.swin = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            img_size=(IMG_HEIGHT, IMG_WIDTH),
            strict_img_size=False,
        )
        self.proj = nn.Linear(512, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        features = self.swin(x)
        feat = features[-2]
        B, H, W, C = feat.shape
        feat = feat.view(B, H*W, C)
        return self.norm(self.proj(feat))

class GPT2Decoder(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL):
        super().__init__()
        config = GPT2Config.from_pretrained("distilgpt2")
        config.add_cross_attention = True
        config.vocab_size = vocab_size
        self.decoder = GPT2LMHeadModel(config)

    def forward(self, input_ids, encoder_hidden_states=None, labels=None):
        return self.decoder(
            input_ids=input_ids,
            encoder_hidden_states=encoder_hidden_states,
            labels=labels
        )

class CompleteHTRPipeline(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL, num_control_points=16):
        super().__init__()
        self.tps_stn = TPSSpatialTransformer(num_control_points)
        self.swin_encoder = SwinEncoder(d_model=d_model)
        self.bilstm = nn.LSTM(
            input_size=d_model, hidden_size=d_model//2, num_layers=2,
            batch_first=True, bidirectional=True, dropout=0.3
        )
        self.gpt2_decoder = GPT2Decoder(vocab_size=vocab_size, d_model=d_model)

    def _extract_visual_memory(self, images):
        rectified = self.tps_stn(images)
        swin_feat = self.swin_encoder(rectified)
        memory, _ = self.bilstm(swin_feat)
        return memory.contiguous()

    def forward(self, images, target_ids, criterion=None):
        memory = self._extract_visual_memory(images)
        dec_input = target_ids[:, :-1].clone()
        dec_input = torch.where(dec_input == -100, torch.ones_like(dec_input), dec_input)
        labels = target_ids[:, 1:].clone()
        outputs = self.gpt2_decoder(input_ids=dec_input, encoder_hidden_states=memory)
        logits = outputs.logits
        if criterion is not None:
            return criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1), ignore_index=-100)

    @torch.no_grad()
    def generate(self, images, max_length=MAX_SEQ_LEN,
                 bos_token_id=0, eos_token_id=2,
                 beam_size=BEAM_SIZE):
        device = images.device
        B = images.size(0)
        memory = self._extract_visual_memory(images)

        if beam_size == 1:
            generated = torch.full((B, 1), bos_token_id, dtype=torch.long, device=device)
            finished = torch.zeros(B, dtype=torch.bool, device=device)
            for _ in range(max_length - 1):
                out = self.gpt2_decoder(input_ids=generated, encoder_hidden_states=memory)
                next_tokens = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
                next_tokens = next_tokens.masked_fill(finished.unsqueeze(1), eos_token_id)
                generated = torch.cat([generated, next_tokens], dim=1)
                finished |= (next_tokens.squeeze(1) == eos_token_id)
                if finished.all():
                    break
            return generated[:, 1:]

        # Beam search (size > 1)
        all_results = []
        for b in range(B):
            mem = memory[b:b+1]
            beams = [(0.0, [bos_token_id])]
            completed = []
            for _ in range(max_length - 1):
                candidates = []
                for score, tokens in beams:
                    if tokens[-1] == eos_token_id:
                        completed.append((score, tokens))
                        continue
                    inp = torch.tensor([tokens], dtype=torch.long, device=device)
                    out = self.gpt2_decoder(input_ids=inp, encoder_hidden_states=mem)
                    log_prob = F.log_softmax(out.logits[0, -1], dim=-1)
                    top_lp, top_id = log_prob.topk(beam_size)
                    for lp, tid in zip(top_lp.tolist(), top_id.tolist()):
                        candidates.append((score + lp, tokens + [tid]))
                if not candidates:
                    break
                candidates.sort(key=lambda x: x[0], reverse=True)
                beams = []
                for score, tokens in candidates[:beam_size * 2]:
                    if tokens[-1] == eos_token_id:
                        completed.append((score, tokens))
                    else:
                        beams.append((score, tokens))
                    if len(beams) == beam_size:
                        break
                if not beams:
                    break
            all_beams = completed if completed else beams
            _, best_tokens = max(all_beams, key=lambda x: x[0])
            result = torch.tensor(best_tokens[1:], dtype=torch.long, device=device)
            all_results.append(result)

        max_len = max(r.size(0) for r in all_results)
        padded = torch.full((B, max_len), eos_token_id, dtype=torch.long, device=device)
        for i, r in enumerate(all_results):
            padded[i, :r.size(0)] = r
        return padded

# =====================================================================
# 4. AGENTIC VERIFICATION (optional, same as training)
# =====================================================================
class AgenticVerificationModule:
    def __init__(self, words_txt_file=None):
        self.lexicon = {}
        self.freq_max = 1
        if words_txt_file and os.path.exists(words_txt_file):
            self._build_lexicon(words_txt_file)

    def _build_lexicon(self, file_path):
        freq = defaultdict(int)
        with open(file_path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) >= 9 and parts[1] == "ok":
                    word = " ".join(parts[8:]).strip().lower()
                    if word:
                        freq[word] += 1
        self.lexicon = dict(freq)
        self.freq_max = max(freq.values()) if freq else 1

    def verify_and_correct(self, text_output, top_logprob=None, logprob_threshold=-0.3):
        cleaned = text_output.strip().lower()
        if (not cleaned or cleaned in self.lexicon or len(cleaned) <= 2
                or any(c.isdigit() for c in cleaned)):
            return text_output

        if text_output[0].isupper() and len(text_output) > 1:
            if text_output[1:].islower() and cleaned in self.lexicon:
                return text_output

        if top_logprob is not None:
            avg_lp = sum(top_logprob) / max(len(top_logprob), 1)
            if avg_lp > logprob_threshold:
                return text_output

        best_candidate = None
        best_score = -float('inf')
        target_len = len(cleaned)

        for word, freq in self.lexicon.items():
            if abs(len(word) - target_len) > 1:
                continue
            dist = Levenshtein.distance(cleaned, word)
            if dist > 1:
                continue
            freq_score = freq / self.freq_max
            score = freq_score - (dist * 0.3)
            if score > best_score:
                best_score = score
                best_candidate = word

        if best_candidate is None:
            return text_output

        if text_output.isupper():
            return best_candidate.upper()
        elif len(text_output) > 0 and text_output[0].isupper():
            return best_candidate.capitalize()
        return best_candidate

# =====================================================================
# 5. METRICS
# =====================================================================
def calculate_metrics(predictions, targets):
    if not targets:
        return {"CER": 0.0, "WER": 0.0}
    total_cer = 0.0
    wrong_words = 0
    for pred, target in zip(predictions, targets):
        p = pred.strip()
        t = target.strip()
        d = Levenshtein.distance(p, t)
        if len(t) > 0:
            total_cer += d / len(t)
        elif len(p) > 0:
            total_cer += 1.0
        if d != 0:
            wrong_words += 1
    n = len(targets)
    return {
        "CER": round((total_cer / n) * 100, 4),
        "WER": round((wrong_words / n) * 100, 4)
    }

# =====================================================================
# 6. MAIN TEST FUNCTION
# =====================================================================
def test():
    print("="*60)
    print("TESTING IAM HTR WITH DISTILGPT-2 (BEAM SEARCH)")
    print("="*60)

    # Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # Tokenizer
    print("\nLoading tokenizer...")
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
    print(f"BOS={tokenizer.bos_token_id} | EOS={tokenizer.eos_token_id} | PAD={tokenizer.pad_token_id}")

    # Test data loader (validation split)
    print("\nCreating test loader (validation split)...")
    test_loader = get_iam_test_loader(BASE_DATA_DIR, WORDS_TXT_FILE, tokenizer, batch_size=BATCH_SIZE)

    # Load model
    print(f"\nLoading model from {CHECKPOINT_PATH}...")
    model = CompleteHTRPipeline(vocab_size=tokenizer.vocab_size).to(device)
    if not os.path.exists(CHECKPOINT_PATH):
        print(f"Checkpoint not found: {CHECKPOINT_PATH}")
        return
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded checkpoint from epoch {checkpoint.get('epoch', '?')} (Best WER: {checkpoint.get('best_wer', '?')}%)")

    # Agentic verification (optional)
    agent_verifier = AgenticVerificationModule(WORDS_TXT_FILE)

    # Inference
    model.eval()
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for images, _, text_labels in tqdm(test_loader, desc=f"Beam search (size={BEAM_SIZE})"):
            images = images.to(device)
            tokens = model.generate(
                images,
                max_length=MAX_SEQ_LEN,
                bos_token_id=tokenizer.bos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                beam_size=BEAM_SIZE
            )
            for i in range(images.size(0)):
                raw = tokenizer.decode(tokens[i], skip_special_tokens=True)
                # optionally apply agentic verification
                verified = agent_verifier.verify_and_correct(raw)
                all_preds.append(verified)
                all_targets.append(text_labels[i])

    # Metrics
    metrics = calculate_metrics(all_preds, all_targets)
    print("\n" + "="*60)
    print("TEST RESULTS")
    print("="*60)
    print(f"Total samples: {len(all_targets)}")
    print(f"Character Error Rate (CER): {metrics['CER']:.2f}%")
    print(f"Word Error Rate (WER): {metrics['WER']:.2f}%")
    print("="*60)

    # Sample predictions
    print("\nSample predictions (first 10):")
    for i in range(min(10, len(all_preds))):
        print(f"GT: {all_targets[i]}")
        print(f"PR: {all_preds[i]}")
        print("-" * 40)

if __name__ == "__main__":
    test()

TESTING IAM HTR WITH DISTILGPT-2 (BEAM SEARCH)
Device: cuda

Loading tokenizer...


/usr/local/lib/python3.8/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


BOS=0 | EOS=2 | PAD=1

Creating test loader (validation split)...
Registered 38305 samples.

Loading model from /home/mca24-26/ocr/iam_htr_distilgpt2.pth...


/usr/local/lib/python3.8/dist-packages/timm/layers/interpolate.py:47: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at ../aten/src/ATen/native/BucketizationUtils.h:32.)
  idx_right = torch.bucketize(x, p)


Loaded checkpoint from epoch 33 (Best WER: 24.5367%)


Beam search (size=5): 100%|█████████████████████████████| 240/240 [47:21<00:00, 11.84s/it]


TEST RESULTS
Total samples: 3831
Character Error Rate (CER): 15.42%
Word Error Rate (WER): 23.83%

Sample predictions (first 10):
GT: any
PR: any
----------------------------------------
GT: any
PR: any
----------------------------------------
GT: unopposed
PR: supporters
----------------------------------------
GT: conference
PR: conference
----------------------------------------
GT: Sir
PR: Sir
----------------------------------------
GT: -
PR: -
----------------------------------------
GT: the
PR: the
----------------------------------------
GT: United
PR: United
----------------------------------------
GT: Mr.
PR: Mr.
----------------------------------------
GT: drastic
PR: despair
----------------------------------------
